In [0]:
Create or replace temp view lookback_npi_5yr AS
select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31') --Date will change here 
where patient_id in ('23CYYEJW'))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    -- WHERE HCP_RANK = 1
))
;

In [0]:
SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31') --Date will change here 
where patient_id in ('23CYYEJW'))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
                

In [0]:
select * from lookback_npi_5yr

In [0]:
select * from (
  SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31') --Date will change here 
where patient_id in (select distinct patient_id from (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
)
where patient_id in (select patient_id
from lookback_npi_5yr
where primary_npi is null)

In [0]:
Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)) --Date will change here 
where patient_id in (select patient_id
from lookback_npi_5yr
where primary_npi is null) and npi is not null

### 5 yr

In [0]:
Create or replace temp view lookback_npi_5yr AS
select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31') --Date will change here 
where patient_id in (select distinct patient_id from (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
;

In [0]:
select *
from lookback_npi_5yr
-- where primary_npi is null

### 4 yr

In [0]:
Create or replace temp view lookback_npi_4yr AS
select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2021-08-01' AND '2025-07-31') --Date will change here 
where patient_id in (select distinct patient_id from (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
;

In [0]:
select *
from lookback_npi_4yr
where primary_npi is not null

### 3 yr

In [0]:
Create or replace temp view lookback_npi_3yr AS
select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2022-08-01' AND '2025-07-31') --Date will change here 
where patient_id in (select distinct patient_id from (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
;

In [0]:
select *
from lookback_npi_3yr
where primary_npi is not null

### 2 yr

In [0]:
Create or replace temp view lookback_npi_2yr AS
select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31') --Date will change here 
where patient_id in (select distinct patient_id from (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
;

In [0]:
select *
from lookback_npi_2yr
where primary_npi is not null

### 1 yr

In [0]:
Create or replace temp view lookback_npi_1yr AS
select distinct n_pats as patient_id, npi as primary_npi
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from (Select * from 
(SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2024-08-01' AND '2025-07-31') --Date will change here 
where patient_id in (select distinct patient_id from (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)))
-- where code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi)
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
;

In [0]:
select *
from lookback_npi_1yr
where primary_npi is not null

### Combining

In [0]:
select *
from lookback_npi_1yr
where primary_npi is not null

### Combining the tables

In [0]:
with patient_cohort as (
  select distinct patient_id from (SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31' and patient_id in (select distinct patient_id from (select distinct PATIENT_ID
from com_edp_prd.com_raw.kom_medical_events
where DIAGNOSIS_CODES ilike '%E761%' and service_date between '2020-08-01' AND '2025-07-31'
union
select distinct PATIENT_ID
from com_edp_prd.com_raw.kom_pharmacy_events
where DIAGNOSIS_CODE = 'E761' and TRANSACTION_STATUS = 'PAID' and fill_date between '2020-08-01' AND '2025-07-31')))
),
joined_table as (select p.patient_id,
e.primary_npi as 1_dx_5_yr_primary_npi,
d.primary_npi as 1_dx_4_yr_primary_npi,
c.primary_npi as 1_dx_3_yr_primary_npi,
b.primary_npi as 1_dx_2_yr_primary_npi,
a.primary_npi as 1_dx_1_yr_primary_npi
from patient_cohort as p
left join lookback_npi_1yr as a on p.patient_id = a.patient_id
left join lookback_npi_2yr as b on p.patient_id = b.patient_id
left join lookback_npi_3yr as c on p.patient_id = c.patient_id
left join lookback_npi_4yr as d on p.patient_id = d.patient_id
left join lookback_npi_5yr as e on p.patient_id = e.patient_id)
select *
from joined_table
where patient_id is not null and (1_dx_5_yr_primary_npi is not null or 1_dx_4_yr_primary_npi is not null or 1_dx_3_yr_primary_npi is not null or 1_dx_2_yr_primary_npi is not null or 1_dx_1_yr_primary_npi is not null)

In [0]:
SELECT *
FROM com_edp_prd.cmpa_insights_internal_schema.lookback_mpsii_elaprase_primary_npi
WHERE patient_id IS NOT NULL
  AND (
    (`5_yr_primary_npi` IS NOT NULL AND lower(trim(`5_yr_primary_npi`)) NOT IN ('', 'null', 'n/a', 'na'))
    OR (`4_yr_primary_npi` IS NOT NULL AND lower(trim(`4_yr_primary_npi`)) NOT IN ('', 'null', 'n/a', 'na'))
    OR (`3_yr_primary_npi` IS NOT NULL AND lower(trim(`3_yr_primary_npi`)) NOT IN ('', 'null', 'n/a', 'na'))
    OR (`2_yr_primary_npi` IS NOT NULL AND lower(trim(`2_yr_primary_npi`)) NOT IN ('', 'null', 'n/a', 'na'))
    OR (`1_yr_primary_npi` IS NOT NULL AND lower(trim(`1_yr_primary_npi`)) NOT IN ('', 'null', 'n/a', 'na'))
  );

